# Multi-Signal Short Squeeze Scanner

Three-stage pipeline combining:
1. **SEC Failure-to-Deliver (FTD) data** — persistent delivery failures as a proxy for hard-to-borrow / heavily shorted stocks
2. **Volume/Price technicals** — volume spikes, Bollinger Band squeeze, price momentum
3. **SEC filing sentiment** — LLM-powered analysis of filings for squeeze catalysts, ownership dynamics, and short seller vulnerability

### Scoring (0-100)
| Signal Group | Max Points | Key Components |
|---|---|---|
| FTD Signals | 30 | FTD ratio, trend, persistence |
| Volume/Price | 35 | Volume spike, BB squeeze, momentum, market cap |
| SEC Sentiment | 35 | Catalyst strength, float pressure, short vulnerability |

The final `buy_signal` is determined by LLM consolidation (not a raw score threshold).

## Setup

In [1]:
import sys, os
import pandas as pd
sys.path.append("../")

from wallstreet_quant.short_squeeze import ShortSqueezeScanner

FMP_KEY = os.getenv('financial_modeling_prep_api_key')
assert FMP_KEY, "Set the financial_modeling_prep_api_key environment variable"

## Load Stock Universe

In [ ]:
universe = pd.read_csv("iShares-Russell-3000-list.csv")
tickers = universe['Ticker'].tolist()
print(f"Universe: {len(tickers)} tickers")

Universe: 2604 tickers


,Ticker,Name,Sector,Asset Class,Market Value,Weight (%),Notional Value,Quantity,Price,Location,Exchange,Currency,FX Rate,Accrual Date
0,NVDA,NVIDIA CORP,Information Technology,Equity,"1,242,041,063.22",6.70,"1,242,041,063.22","6,518,874.00",190.53,United States,NASDAQ,USD,1.0,--
1,AAPL,APPLE INC,Information Technology,Equity,"1,110,193,739.60",5.99,"1,110,193,739.60","4,060,694.00",273.40,United States,NASDAQ,USD,1.0,--
2,MSFT,MICROSOFT CORP,Information Technology,Equity,"1,007,480,104.56",5.44,"1,007,480,104.56","2,065,736.00",487.71,United States,NASDAQ,USD,1.0,--
3,AMZN,AMAZON COM INC,Consumer Discretionary,Equity,"621,264,840.04",3.35,"621,264,840.04","2,671,877.00",232.52,United States,NASDAQ,USD,1.0,--
4,GOOGL,ALPHABET INC CLASS A,Communication,Equity,"507,593,068.15",2.74,"507,593,068.15","1,619,065.00",313.51,United States,NASDAQ,USD,1.0,--


## Initialize Scanner

In [3]:
scanner = ShortSqueezeScanner(
    fmp_api_key=FMP_KEY,
    batch_delay=0.25,       # seconds between FMP API calls
    model="gpt-5.2-pro"     # model for SEC sentiment analysis
)

## Run Full Scan

**Stage 1** (all tickers): Compute FTD + volume/price signals. This is cheap — only FMP API calls + SEC FTD downloads.

**Stage 2** (filtered only): Tickers passing `ftd_score_threshold` get SEC filing sentiment analysis via 3 specialized LLM calls each.

**Stage 3** (filtered only): LLM consolidation produces a final assessment + buy/no-buy signal.

Adjust `ftd_score_threshold` to control how many tickers reach the expensive LLM stage. Lower = more candidates but higher cost.

In [4]:
results = scanner.scan(
    tickers,
    ftd_score_threshold=50,    # min (ftd_score + vp_score) to reach Stage 2
    run_sec_sentiment=True     # set False to skip LLM analysis (Stage 1 only)
)

SHORT SQUEEZE SCANNER — 2604 tickers

[Stage 1] Loading SEC FTD data...
  FTD data loaded: 271777 records across 3 months
[Stage 1] Computing FTD + volume/price signals for 2604 tickers...
  Progress: 0/2604 (0 processed)



1 Failed download:
['BRKB']: YFPricesMissingError('possibly delisted; no price data found  (period=75d) (Yahoo error = "No data found, symbol may be delisted")')


  Progress: 50/2604 (50 processed)
  Progress: 100/2604 (100 processed)
  Progress: 150/2604 (150 processed)



1 Failed download:
['XTSLA']: HTTPError('HTTP Error 404: ')


  Progress: 200/2604 (200 processed)
  Progress: 250/2604 (250 processed)
  Progress: 300/2604 (300 processed)
  Progress: 350/2604 (350 processed)
  Progress: 400/2604 (400 processed)


...

  Progress: 2600/2604 (2600 processed)



1 Failed download:
['--']: YFPricesMissingError('possibly delisted; no price data found  (period=75d) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['--']: YFPricesMissingError('possibly delisted; no price data found  (period=75d) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['RTYH6']: YFPricesMissingError('possibly delisted; no price data found  (period=75d) (Yahoo error = "No data found, symbol may be delisted")')

1 Failed download:
['ESH6']: YFPricesMissingError('possibly delisted; no price data found  (period=75d) (Yahoo error = "No data found, symbol may be delisted")')



[Stage 1] Complete. 2/2604 tickers passed threshold (50).

[Stage 2] Running SEC sentiment analysis on 2 candidates...
  [1/2] Analyzing PCT...


  [2/2] Analyzing HRTX...



[Stage 3] Generating assessments for 2 candidates...

Scan complete. 0 buy signals out of 2 candidates.


## View Buy Candidates

In [10]:
candidates = results[results['buy_signal'] == True].sort_values('composite_score', ascending=False)
print(f"Buy candidates: {len(candidates)}")
display(candidates[['ticker', 'composite_score', 'ftd_score', 'vp_score', 'sec_score',
                     'assessment', 'buy_signal']])

Buy candidates: 0


,ticker,composite_score,ftd_score,vp_score,sec_score,assessment,buy_signal


## View All Screened Tickers

In [12]:
display(results[['ticker', 'price', 'composite_score', 'ftd_score', 'ftd_trend',
                  'vp_score', 'bb_squeeze', 'volume_spike_ratio',
                  'catalyst_strength', 'float_pressure', 'vulnerability_level',
                  'sec_score', 'assessment', 'buy_signal']].head(30))
print(f"Total screened: {len(results)}")

,ticker,price,composite_score,ftd_score,ftd_trend,vp_score,bb_squeeze,volume_spike_ratio,catalyst_strength,float_pressure,vulnerability_level,sec_score,assessment,buy_signal
0,HRTX,0.853,71.44,29.95,increasing,26.15,True,1.23,moderate,none,moderate,15.34,HRTX shows only a mild uptick in fails-to-deli...,False
1,PCT,5.940,56.00,30.00,increasing,22.00,True,0.74,none,low,low-to-moderate,4.00,PCT shows a moderately elevated FTD level with...,False


Total screened: 2


## Save to Excel

In [7]:
filepath = scanner.save_results(results)
print(f"Saved to: {filepath}")

Results saved to: ./short_squeeze_2026-04-12.xlsx
Saved to: ./short_squeeze_2026-04-12.xlsx


## Debug: Single-Stock Analysis

Examine all signal components for a specific ticker.

In [8]:
# Pick a ticker to inspect
test_ticker = "GME"

# FTD signals
ftd_df = scanner._load_recent_ftd_data()
ftd_signals = scanner.compute_ftd_signals(test_ticker, ftd_df)
print("=== FTD Signals ===")
for k, v in ftd_signals.items():
    print(f"  {k}: {v}")

# Volume/Price signals
vp_signals = scanner.compute_volume_price_signals(test_ticker)
print("\n=== Volume/Price Signals ===")
for k, v in vp_signals.items():
    print(f"  {k}: {v}")

# SEC sentiment
sec_signals = scanner.compute_sec_sentiment(test_ticker)
print("\n=== SEC Sentiment ===")
for k, v in sec_signals.items():
    print(f"  {k}: {v}")

# Composite
composite = ftd_signals['ftd_score'] + vp_signals['vp_score'] + sec_signals['sec_score']
print(f"\nComposite Score: {composite:.1f}/100")

=== FTD Signals ===
  ftd_total_fails: 2191934
  ftd_avg_daily_fails: 62626.7
  ftd_max_daily_fails: 359757
  ftd_days_with_fails: 35
  ftd_trend: stable
  ftd_ratio: 0.0
  ftd_score: 10.0

=== Volume/Price Signals ===
  price: 23.22
  market_cap: 10411267500
  avg_volume: 7380549
  volume_spike_ratio: 0.79
  relative_volume_5d: 0.81
  price_momentum_5d: -0.6
  price_momentum_10d: 2.93
  bb_width: 0.0703
  bb_squeeze: False
  vp_score: 0.0



=== SEC Sentiment ===
  catalyst_strength: moderate
  catalyst_summary: The filing contains a real, material operating turnaround (profitability + margin expansion + large SG&A reduction), plus evidence of a higher-margin collectibles-driven mix shift and several collectibles-related product/service rollouts (including Power Packs). Liquidity is extremely large and could enable an M&A/control-transaction catalyst, but timing and targets are unspecified, so that component is weaker as a squeeze trigger.
  float_pressure: low
  ownership_summary: This excerpt contains no buyback authorization/activity and no open-market insider buying or institutional accumulation signals. The only potentially supportive squeeze-related ownership dynamic is the unquantified note that Cohen is the largest stockholder (some concentration). However, the dominant theme is prospective float expansion/dilution (convertible notes, large warrant overhang, potential additional warrant issuance, possible ATM/equi